# What value do the datasets have? (drafting)

## Assumptions

### Duplicates

In 01_data_quality.ipynb you can see each dataset has a duplicate problem. I would prefer to do more research on best way to select the best record for duplicates, but for now going on some quick way to rank based of assuming I understand the data correctly.

Each data source, I set up a staging table to clean up some fields, provide clean and normalized company name & address, and calculate a duplicate record rank (`dupe_record_rank`). 

This rank field allows me to quickly filter to `dupe_record_rank=1` to remove the more obvious duplicates. 

Each one is partitioned the same:
``` sql
    partition by 
        company_name_clean,
        street_address_clean,
        city_clean,
        state_clean,
        zip_code
```
How the sort happened varied per dataset, which can be seen below:

#### card_list
``` sql
    rank() over (
        partition by 
            lower(trim(dba_name)),
            lower(trim(addr1)),
            lower(trim(city)),
            lower(trim(state)),
            zip_code
        order by 
            merchant_outlet_number desc
    )
```
This one I'm making the assumption that `merchant_outlet_number` is added in numerical order (larger the id value, it's more recent addition) and you want the more recent one.

#### purchased_list
``` sql
    rank() over (
        partition by 
            lower(trim("name")),
            lower(trim(street)),
            lower(trim(city)),
            lower(trim(state)),
            postal_code
        order by 
            speedeonid desc
    )
```
This one I'm making the assumption that `speedeonid` is added in numerical order (larger the id value, it's more recent addition) and you want the more recent one.

#### salesforce_list
``` sql
    rank() over (
        partition by 
            lower(trim(company)),
            lower(trim(street)),
            lower(trim(city)),
            lower(trim(state)),
            zip5__c
        order by 
            status!='Disqualified' desc, 
            total_snap_opportunity_tso__c is not null desc, 
            zip_code_raw is not null desc, 
            industry_vertical__c is not null desc, 
            id desc
    )
```
For this one, I figured that you wouldn't want to look at Disqualified accounts, prefer ones that have an opportunity value, complete address, meta data is there, and finally use the id like above.


#### snap
``` sql
    rank() over (
        partition by 
            lower(trim(dba_name)),
            lower(trim(merchant_address)),
            lower(trim(merchant_city)),
            lower(trim(merchant_state)),
            merchant_zip_code
        order by 
            is_inactivated, 
            mid::int desc
    )
```
I ranked the active accounts higher then use the id field like others above.

#### website_list
``` sql
    rank() over (
        partition by 
            lower(trim(business_name)),
            lower(trim(street_address)),
            lower(trim(city)), 
            lower(trim(state)),
            zip_code
        order by 
            street_address_raw is not null desc, 
            industry_name is not null desc, 
            industry_subcategory!='OTHER' desc, 
            lead_id desc, 
            industry_subcategory
    )
```
This dataset does not have an unique id. So if going to production, would definitely work on creating unique rows and an ID, but for now some ranking. The ranking looked for complete address, industry data is field, more specific industry value, the lead id, and then industry subcategory has a value. 

## Setup

In [1]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
import plotly.express as px
import kaleido

# Find the repo root by walking up until the warehouse folder is found,
# so the notebook works whatever folder the kernel starts in.
here = Path.cwd().resolve()
repo_root = next(p for p in [here, *here.parents] if (p / "warehouse" / "snap-assessment.duckdb").exists())

# read_only=True so the notebook never blocks `dbt build` from writing to the file.
con = duckdb.connect(str(repo_root / "warehouse" / "snap-assessment.duckdb"), read_only=True)

def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return the results as a Pandas DataFrame."""
    return con.sql(sql).df()

## Values per Table

## snap

Let's look at the internal list to see if we can find some similarities of accounts that have been successful for Snap.

Reminder: using the `stg_snap` table, that clean & normalized fields and rank to remove dupe records (under [Assumptions](#assumptions)).

**product type**

In [2]:
q("""

WITH merchant_segments AS (
    SELECT
        product_type,
        COUNT(DISTINCT mid) AS active_merchants,
        count(distinct industry) as n_industries,
        SUM(apps_ytd) AS apps_ytd,
        SUM(aprv_ytd) AS aprv_ytd,
        SUM(cmpl_ytd) AS cmpl_ytd,
        SUM(fund_ytd) AS fund_ytd
    FROM "snap-assessment".staging.stg_snap
    WHERE NOT is_inactivated
      AND dupe_record_rank = 1
    GROUP BY ALL
)

SELECT
    product_type,

    -- Merchant coverage
    active_merchants,
    n_industries,

    -- Application volume
    apps_ytd,

    -- Conversion
    ROUND(100.0 * aprv_ytd / NULLIF(apps_ytd, 0), 2) AS approval_rate,
    ROUND(100.0 * cmpl_ytd / NULLIF(apps_ytd, 0), 2) AS application_completion_rate,
    ROUND(100.0 * cmpl_ytd / NULLIF(aprv_ytd, 0), 2) AS approved_completion_rate,

    -- Funding
    fund_ytd as total_funding_ytd,
    ROUND(fund_ytd / NULLIF(active_merchants, 0), 2) AS avg_funding_per_merchant

FROM merchant_segments
ORDER BY product_type, fund_ytd DESC

""")

,product_type,active_merchants,n_industries,apps_ytd,approval_rate,application_completion_rate,approved_completion_rate,total_funding_ytd,avg_funding_per_merchant
0,Lease,425,11,34893.0,60.28,39.92,66.22,20632661.0,48547.44
1,Loan,229,6,6452.0,51.49,34.61,67.22,3534148.0,15432.96


**product and industry**

In [3]:
df_snap_performance = q("""

SELECT
    product_type,
    industry,
    COUNT(DISTINCT mid) AS active_merchants,
    SUM(apps_ytd) AS apps_ytd,
    SUM(aprv_ytd) AS aprv_ytd,

    ROUND(
        100.0 * SUM(aprv_ytd) / NULLIF(SUM(apps_ytd), 0), 2
    ) AS approval_rate,

    SUM(cmpl_ytd) AS cmpl_ytd,

    ROUND(
        100.0 * SUM(cmpl_ytd) / NULLIF(SUM(apps_ytd), 0), 2
    ) AS completion_rate,

    SUM(fund_ytd) AS fund_ytd

FROM "snap-assessment".staging.stg_snap
WHERE NOT is_inactivated
  AND dupe_record_rank = 1
GROUP BY ALL
HAVING SUM(COALESCE(fund_ytd, 0)) > 0
ORDER BY apps_ytd DESC;

""")

In [ ]:
fig = px.scatter(
    df_snap_performance,
    x="apps_ytd",
    y="approval_rate",
    size="fund_ytd",  ##size of the bubble is the total funding YTD
    color="product_type",  ##color of the bubble is the product type
    hover_name="industry", ##hover over the bubble to see the industry name
    hover_data={
        "active_merchants": ":,",
        "apps_ytd": ":,",
        "aprv_ytd": ":,",
        "approval_rate": ":.2f",
        "cmpl_ytd": ":,",
        "completion_rate": ":.2f",
        "fund_ytd": ":$,.0f"
    },
    size_max=45,
    title="Snap Merchant Performance by Industry and Product",
    labels={
        "apps_ytd": "Applications YTD",
        "approval_rate": "Approval Rate (%)",
        "product_type": "Product Type",
        "fund_ytd": "Funding YTD"
    }
)

fig.update_layout(
    template="plotly_white",
    height=650,
    legend_title_text="Product Type"
)

fig.update_yaxes(ticksuffix="%")

fig.show()

In case the image doesn't show while browsing the repo:

![Snap Merchant Performance](../figures/snap_industry_performance.png)

created using `fig.write_image("figures/snap_industry_performance.png")`

**How I'd interpret the plot above**

- High volume / high approval
    - Strong market
    - Established segments with demonstrated application demand and conversion.

- Low volume / high approval
    - Explore growth
    - Potentially underserved segments worth investigating.

- High volume / low approval
    - Investigate
    - Demand exists, but approval performance warrants further investigation.

- Low volume / low approval
    - Lower priority
    - Less demonstrated demand and weaker conversion.

One caveat: low approval rates don't necessarily mean an industry is unattractive. They could reflect customer credit profiles, application quality, or differences in Snap's underwriting criteria. I'd avoid treating approval rate as a direct measure of merchant quality.

In [ ]:
df_snap_performance

,product_type,industry,active_merchants,apps_ytd,aprv_ytd,approval_rate,cmpl_ytd,completion_rate,fund_ytd
0,Lease,FURNITURE,98,9731.0,5809.0,59.70,3753.0,38.57,5476968.0
1,Lease,TIRES_RIMS,159,9395.0,6432.0,68.46,4692.0,49.94,6660048.0
2,Loan,AUTO_SERVICE,211,5758.0,3002.0,52.14,2025.0,35.17,3246507.0
3,Lease,JEWELRY,36,4228.0,2156.0,50.99,1374.0,32.50,2474761.0
4,Lease,OTHER,11,4067.0,1899.0,46.69,996.0,24.49,1638829.0
5,Lease,AUTO_SERVICE,39,2380.0,1718.0,72.18,1206.0,50.67,1826549.0
6,Lease,APPLIANCES,24,1682.0,1113.0,66.17,745.0,44.29,1021383.0
7,Lease,CAR_AUDIO,36,1471.0,937.0,63.70,607.0,41.26,916177.0
8,Lease,MUSICAL_INSTRUMENTS,1,939.0,444.0,47.28,209.0,22.26,188545.0
9,Lease,MATTRESSES,15,740.0,380.0,51.35,290.0,39.19,330061.0


Auto Service and Tires industries seem to have worked real well for Snap.

I think some work could also be done to explore if location is a factor, but with the addresses all over the place. I'm going to place this on hold.

## card_list

The card list dataset had the field, `last_activity_period`, to help look at freshness of company. Definitely want to prioritize a company that has more recent transactions when prospecting.

In [ ]:
q("""

select
  last_activity_period,
  count(*) as n_companies
from "snap-assessment".staging.stg_card_list
where 1=1
  and dupe_record_rank=1
group by all
order by n_companies desc

""")

,last_activity_period,n_companies
0,Txn within 13 months,1044
1,Txn between 14-24 months ago,87
2,Reg inactive with txn between 3 and 13 months,58
3,Txn between 2-3 years ago,44
4,Txn before 3 years ago,40
5,No Txn,38
6,Reg inactive with txn in last 3 months,3


Definitely prioritize the companies within 'Txn within 13 months'.

More work could be done to get the `mcc_id` by getting a list of `mcc_id` codes and values, to better prioritize industries in the auto & tire world seeing that is a popular industry currently for Snap.

## purchased_list

The `purchased_list` was one of the cleanest datasets.

- It could be used to clean up address of companies: prospects and already in data
- Provides more details about the industry.
- Then some meta data about the employee count for the location and size of building.

## salesforce_list

In [ ]:
q("""

select
	status,
	count(*) as n_companies
from "snap-assessment".staging.stg_salesforce_list
where dupe_record_rank=1
group by all
order by n_companies desc

""")

,status,n_companies
0,Disqualified,616
1,Converted,466
2,Nurturing,173
3,Working,5
4,Merchant Invitation Sent,4
5,MQL,3
6,BDR Working,1
7,New,1
8,Submit to Join,1
9,Awaiting Approval,1


This data definitely helps narrow down to accounts where they aren't disqualified or already converted. Then could possibly look at industries that we know work well with Snap already (auto/tires). The age of lead also helps with how long they have been sitting in salesforce.

## website_list

This dataset is the messiest, which makes sense as a user is filling out some form. This does mean that they are showing some interest though, but need to check if already on the system. Or maybe it's the same company but new location, which could be more likely as a convert if company already a customer.

In [ ]:
con.close()